In [1]:
# Data processing
import pandas as pd
import numpy as np
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

# Deep Learning (GPU)
import tensorflow as tf
import keras

from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam

I0000 00:00:1794120816.162044     648 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794120816.352889     648 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794120817.891670     648 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
print("TensorFlow version:", tf.__version__)
print("Keras backend:", keras.config.backend())
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.22.0-rc0
Keras backend: tensorflow
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


W0000 00:00:1794120821.384846     648 gpu_device.cc:2453] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


In [3]:
with tf.device("/GPU:0"):
    a = tf.random.normal((1000, 1000))
    b = tf.random.normal((1000, 1000))
    c = tf.matmul(a, b)

print("Calculation device:", c.device)

Calculation device: /job:localhost/replica:0/task:0/device:GPU:0


W0000 00:00:1794120822.965268     648 gpu_device.cc:2453] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1794120823.102090     648 gpu_device.cc:2036] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9217 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070, pci bus id: 0000:02:00.0, compute capability: 12.0a
E0000 00:00:1794120823.152719     648 cuda_executor.cc:1365] [0] Failed to allocate device memory of 9.00GiB (9664724992 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:45
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:230
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:487

E0000 00:00:1794120823.154715     648 cuda_executor.cc:1365] [0] Failed to allocate device memory of 8.10GiB (869825

In [4]:
data_path = Path(
    "/mnt/c/Users/Computia.ME/Desktop/Desktop/GA/Capstone/"
    "Sleep Monitor/Data/Data_2/"
    "a-multi-night-instantaneous-heart-rate-and-accelerometry-dataset-with-eeg-sleep-stage-labels-1.0.1"
)

print("Dataset found:", data_path.exists())

Dataset found: True


In [5]:
for folder in sorted(data_path.iterdir()):
    print(folder.name)

Bidslab00
Bidslab01
Bidslab02
Bidslab06
Bidslab07
Bidslab08
Bidslab09
Bidslab10
Bidslab11
Bidslab13
Bidslab14
Bidslab15
Bidslab16
Bidslab17
Bidslab18
Bidslab19
Bidslab20
Bidslab22
Bidslab30
Bidslab31
Bidslab32
Bidslab34
Bidslab35
Bidslab36
Bidslab38
Bidslab39
Bidslab40
Bidslab41
Bidslab42
Bidslab43
Bidslab44
Bidslab45
Bidslab47
Bidslab49
Bidslab50
Bidslab51
Bidslab52
Bidslab53
Bidslab55
Bidslab56
Bidslab60
Bidslab62
Bidslab63
Bidslab64
Bidslab65
Bidslab66
Bidslab68
LICENSE.txt
README.md
SHA256SUMS.txt


In [6]:
from scipy.io import loadmat

# Select one participant and night
night_path = data_path / "Bidslab00" / "1"

# Check files
print("Files:", [f.name for f in night_path.iterdir()])

# Inspect CSV format before loading
print("HR first line:",
      (night_path / "hr.csv").open().readline().strip())

print("Motion first line:",
      (night_path / "motion.csv").open().readline().strip())

# Read labels
labels = loadmat(night_path / "labels.mat")

print("Label variables:",
      [k for k in labels if not k.startswith("__")])

print("Expert labels shape:", labels["expert_label"].shape)
print("Recording start:", labels["recStart"])

Files: ['hr.csv', 'labels.mat', 'motion.csv']
HR first line: 1638504593.0297956,69.0
Motion first line: Timestamp,x,y,z
Label variables: ['dreem_label', 'expert_label', 'recStart']
Expert labels shape: (1, 935)
Recording start: ['2021-12-02 23:11:25']


In [7]:
# Heart rate (no header)
hr = pd.read_csv(
    night_path / "hr.csv",
    header=None,
    names=["time", "hr"]
)

# Accelerometer (has a header)
motion = pd.read_csv(
    night_path / "motion.csv"
)

# Expert-reviewed sleep labels
sleep_labels = pd.DataFrame({
    "sleep_stage": labels["expert_label"].flatten()
})

In [8]:
print("Heart Rate:", hr.shape)
print("Motion:", motion.shape)
print("Labels:", sleep_labels.shape)

display(hr.head())
display(motion.head())
display(sleep_labels.head())

Heart Rate: (4791, 2)
Motion: (1401000, 4)
Labels: (935, 1)


,time,hr
0,1.638505e+09,69.0
1,1.638505e+09,68.0
2,1.638505e+09,67.0
3,1.638505e+09,67.0
4,1.638505e+09,66.0


,Timestamp,x,y,z
0,1.638505e+09,-0.096252,-0.185989,-0.964371
1,1.638505e+09,-0.098297,-0.188156,-0.969070
2,1.638505e+09,-0.081436,-0.173294,-0.983490
3,1.638505e+09,-0.095596,-0.177948,-0.982468
4,1.638505e+09,-0.087845,-0.171478,-0.982178


,sleep_stage
0,0
1,0
2,0
3,0
4,0


In [9]:
stage_names = {
    0: "Wake",
    1: "N1",
    2: "N2",
    3: "N3",
    4: "REM",
    5: "Unknown"
}

sleep_labels["stage_name"] = sleep_labels["sleep_stage"].map(stage_names)

print(sleep_labels["stage_name"].value_counts())

stage_name
N2      347
N3      273
REM     248
Wake     36
N1       31
Name: count, dtype: int64


In [15]:
print("Heart Rate:", hr.shape)
print("Motion:", motion.shape)

display(hr.head())
display(motion.head())

print("\nMissing heart rate values:")
print(hr.isna().sum())

print("\nMissing motion values:")
print(motion.isna().sum())

Heart Rate: (4776, 3)
Motion: (1396135, 6)


,time,hr,epoch
15,1.638505e+09,65.0,0
16,1.638505e+09,63.0,0
17,1.638505e+09,63.0,0
18,1.638505e+09,63.0,0
19,1.638505e+09,64.0,0


,Timestamp,x,y,z,epoch,magnitude
4865,1.638505e+09,-0.656647,0.725723,-0.110886,0,0.984964
4866,1.638505e+09,-0.668381,0.751511,-0.092468,0,1.009976
4867,1.638505e+09,-0.661407,0.752625,-0.099274,0,1.006856
4868,1.638505e+09,-0.653992,0.765686,-0.100479,0,1.011967
4869,1.638505e+09,-0.661179,0.781311,-0.099854,0,1.028385



Missing heart rate values:
time     0
hr       0
epoch    0
dtype: int64

Missing motion values:
Timestamp    0
x            0
y            0
z            0
epoch        0
magnitude    0
dtype: int64


In [11]:
# Convert recording start time to Unix timestamp
rec_start = (
    pd.Timestamp(labels["recStart"].item())
    .tz_localize("America/New_York")
    .timestamp()
)

# Assign each measurement to a 30-second period
hr["epoch"] = np.floor(
    (hr["time"] - rec_start) / 30
).astype(int)

motion["epoch"] = np.floor(
    (motion["Timestamp"] - rec_start) / 30
).astype(int)

# Keep only measurements within the labeled recording
n_epochs = len(sleep_labels)

hr = hr[hr["epoch"].between(0, n_epochs - 1)].copy()
motion = motion[motion["epoch"].between(0, n_epochs - 1)].copy()

# Check alignment
print("Total labeled epochs:", n_epochs)
print("Epochs with heart rate:", hr["epoch"].nunique())
print("Epochs with motion:", motion["epoch"].nunique())

display(hr.head())
display(motion.head())

Total labeled epochs: 935
Epochs with heart rate: 934
Epochs with motion: 934


,time,hr,epoch
15,1.638505e+09,65.0,0
16,1.638505e+09,63.0,0
17,1.638505e+09,63.0,0
18,1.638505e+09,63.0,0
19,1.638505e+09,64.0,0


,Timestamp,x,y,z,epoch
4865,1.638505e+09,-0.656647,0.725723,-0.110886,0
4866,1.638505e+09,-0.668381,0.751511,-0.092468,0
4867,1.638505e+09,-0.661407,0.752625,-0.099274,0
4868,1.638505e+09,-0.653992,0.765686,-0.100479,0
4869,1.638505e+09,-0.661179,0.781311,-0.099854,0


In [12]:
# Heart rate features
hr_features = hr.groupby("epoch").agg(
    hr_mean=("hr", "mean"),
    hr_std=("hr", "std"),
    hr_min=("hr", "min"),
    hr_max=("hr", "max")
).reset_index()

# Calculate movement magnitude
motion["magnitude"] = np.sqrt(
    motion["x"]**2 +
    motion["y"]**2 +
    motion["z"]**2
)

# Movement features
motion_features = motion.groupby("epoch").agg(
    motion_mean=("magnitude", "mean"),
    motion_std=("magnitude", "std"),
    motion_min=("magnitude", "min"),
    motion_max=("magnitude", "max")
).reset_index()

display(hr_features.head())
display(motion_features.head())

,epoch,hr_mean,hr_std,hr_min,hr_max
0,0,64.333333,1.966384,63.0,68.0
1,1,69.333333,1.527525,68.0,71.0
2,2,74.000000,1.000000,73.0,75.0
3,3,76.666667,3.214550,73.0,79.0
4,4,72.000000,0.000000,72.0,72.0


,epoch,motion_mean,motion_std,motion_min,motion_max
0,0,1.005263,0.083865,0.597759,1.707987
1,1,1.016832,0.136838,0.334249,2.314065
2,2,1.048849,0.246704,0.263571,3.822546
3,3,1.052265,0.263902,0.115942,3.001449
4,4,1.032463,0.199030,0.374140,2.522472


In [13]:
# Add epoch numbers to the labels
sleep_labels["epoch"] = np.arange(len(sleep_labels))

# Merge heart rate and motion features
df = hr_features.merge(
    motion_features,
    on="epoch",
    how="inner"
)

# Add sleep-stage labels
df = df.merge(
    sleep_labels,
    on="epoch",
    how="inner"
)

# Check the result
print("Final shape:", df.shape)
display(df.head())

Final shape: (934, 11)


,epoch,hr_mean,hr_std,hr_min,hr_max,motion_mean,motion_std,motion_min,motion_max,sleep_stage,stage_name
0,0,64.333333,1.966384,63.0,68.0,1.005263,0.083865,0.597759,1.707987,0,Wake
1,1,69.333333,1.527525,68.0,71.0,1.016832,0.136838,0.334249,2.314065,0,Wake
2,2,74.000000,1.000000,73.0,75.0,1.048849,0.246704,0.263571,3.822546,0,Wake
3,3,76.666667,3.214550,73.0,79.0,1.052265,0.263902,0.115942,3.001449,0,Wake
4,4,72.000000,0.000000,72.0,72.0,1.032463,0.199030,0.374140,2.522472,0,Wake


In [14]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nSleep stages:")
print(df["stage_name"].value_counts())

Shape: (934, 11)

Missing values:
epoch          0
hr_mean        0
hr_std         1
hr_min         0
hr_max         0
motion_mean    0
motion_std     0
motion_min     0
motion_max     0
sleep_stage    0
stage_name     0
dtype: int64

Sleep stages:
stage_name
N2      347
N3      273
REM     248
Wake     35
N1       31
Name: count, dtype: int64


# Defining Functions to Process the all Nights for all Participants

In [17]:
def process_night(night_path):

    # 1. Read the files
    hr = pd.read_csv(
        night_path / "hr.csv",
        header=None,
        names=["time", "hr"]
    )

    motion = pd.read_csv(night_path / "motion.csv")

    labels = loadmat(night_path / "labels.mat")

    # 2. Get recording start time
    rec_start = (
        pd.Timestamp(labels["recStart"].item())
        .tz_localize("America/New_York")
        .timestamp()
    )

    # 3. Prepare sleep-stage labels
    sleep_labels = pd.DataFrame({
        "epoch": np.arange(labels["expert_label"].size),
        "sleep_stage": labels["expert_label"].flatten()
    })

    # Remove unknown labels
    sleep_labels = sleep_labels[
        sleep_labels["sleep_stage"].isin([0, 1, 2, 3, 4])
    ]

    # 4. Assign epochs
    hr["epoch"] = np.floor(
        (hr["time"] - rec_start) / 30
    ).astype(int)

    motion["epoch"] = np.floor(
        (motion["Timestamp"] - rec_start) / 30
    ).astype(int)

    n_epochs = labels["expert_label"].size

    hr = hr[hr["epoch"].between(0, n_epochs - 1)]
    motion = motion[motion["epoch"].between(0, n_epochs - 1)].copy()

    # 5. Extract heart-rate features
    hr_features = hr.groupby("epoch").agg(
        hr_mean=("hr", "mean"),
        hr_std=("hr", "std"),
        hr_min=("hr", "min"),
        hr_max=("hr", "max"),
        hr_count=("hr", "count")
    ).reset_index()

    # 6. Calculate acceleration magnitude
    motion["magnitude"] = np.sqrt(
        motion["x"]**2 +
        motion["y"]**2 +
        motion["z"]**2
    )

    # 7. Extract movement features
    motion_features = motion.groupby("epoch").agg(
        motion_mean=("magnitude", "mean"),
        motion_std=("magnitude", "std"),
        motion_min=("magnitude", "min"),
        motion_max=("magnitude", "max")
    ).reset_index()

    # 8. Merge everything
    df = sleep_labels.merge(
        hr_features, on="epoch", how="inner"
    )

    df = df.merge(
        motion_features, on="epoch", how="inner"
    )

    # 9. Handle missing standard deviations
    df["hr_std"] = df["hr_std"].fillna(0)
    df["motion_std"] = df["motion_std"].fillna(0)

    return df

In [18]:
test_df = process_night(
    data_path / "Bidslab00" / "1"
)

print("Shape:", test_df.shape)

print("\nMissing values:")
print(test_df.isna().sum())

print("\nSleep-stage distribution:")
print(test_df["sleep_stage"].value_counts())

display(test_df.head())

Shape: (934, 11)

Missing values:
epoch          0
sleep_stage    0
hr_mean        0
hr_std         0
hr_min         0
hr_max         0
hr_count       0
motion_mean    0
motion_std     0
motion_min     0
motion_max     0
dtype: int64

Sleep-stage distribution:
sleep_stage
2    347
3    273
4    248
0     35
1     31
Name: count, dtype: int64


,epoch,sleep_stage,hr_mean,hr_std,hr_min,hr_max,hr_count,motion_mean,motion_std,motion_min,motion_max
0,0,0,64.333333,1.966384,63.0,68.0,6,1.005263,0.083865,0.597759,1.707987
1,1,0,69.333333,1.527525,68.0,71.0,3,1.016832,0.136838,0.334249,2.314065
2,2,0,74.000000,1.000000,73.0,75.0,3,1.048849,0.246704,0.263571,3.822546
3,3,0,76.666667,3.214550,73.0,79.0,3,1.052265,0.263902,0.115942,3.001449
4,4,0,72.000000,0.000000,72.0,72.0,3,1.032463,0.199030,0.374140,2.522472


In [20]:
all_nights = []
failed_nights = []

for person_folder in sorted(data_path.glob("Bidslab*")):

    if not person_folder.is_dir():
        continue

    for night_folder in sorted(person_folder.iterdir()):

        if not night_folder.is_dir():
            continue

        # Check that all three files exist
        required = ["hr.csv", "motion.csv", "labels.mat"]

        if not all((night_folder / f).exists() for f in required):
            continue

        try:
            # Process one night
            night_df = process_night(night_folder)

            # Keep participant and night information
            night_df["person"] = person_folder.name
            night_df["night"] = night_folder.name

            all_nights.append(night_df)

            print(
                person_folder.name,
                "Night", night_folder.name,
                "-", len(night_df), "epochs"
            )

        except Exception as e:
            failed_nights.append(
                (person_folder.name, night_folder.name, str(e))
            )
            print("ERROR:", person_folder.name, night_folder.name, e)

# Combine all nights into one DataFrame
if all_nights:
    combined_df = pd.concat(all_nights, ignore_index=True)
else:
    raise ValueError("No nights were processed successfully.")

Bidslab00 Night 1 - 934 epochs
Bidslab00 Night 2 - 941 epochs
Bidslab00 Night 3 - 991 epochs
Bidslab00 Night 4 - 885 epochs
Bidslab00 Night 5 - 860 epochs
Bidslab00 Night 6 - 885 epochs
Bidslab00 Night 7 - 664 epochs
Bidslab01 Night 1 - 859 epochs
Bidslab01 Night 2 - 880 epochs
Bidslab01 Night 3 - 609 epochs
Bidslab01 Night 4 - 760 epochs
Bidslab01 Night 5 - 897 epochs
Bidslab02 Night 1 - 869 epochs
Bidslab02 Night 2 - 849 epochs
Bidslab02 Night 3 - 829 epochs
Bidslab02 Night 4 - 804 epochs
Bidslab02 Night 5 - 741 epochs
Bidslab02 Night 6 - 871 epochs
Bidslab06 Night 1 - 833 epochs
Bidslab06 Night 2 - 852 epochs
Bidslab06 Night 3 - 720 epochs
Bidslab06 Night 4 - 912 epochs
Bidslab07 Night 1 - 1065 epochs
Bidslab07 Night 2 - 891 epochs
Bidslab07 Night 3 - 860 epochs
Bidslab07 Night 4 - 979 epochs
Bidslab07 Night 5 - 833 epochs
Bidslab07 Night 6 - 757 epochs
Bidslab07 Night 7 - 757 epochs
Bidslab08 Night 1 - 829 epochs
Bidslab08 Night 2 - 951 epochs
Bidslab08 Night 3 - 758 epochs
Bidslab

In [21]:
print("Combined shape:", combined_df.shape)

print("Participants:", combined_df["person"].nunique())

print(
    "Total nights:",
    combined_df.groupby(["person", "night"]).ngroups
)

print("Failed nights:", len(failed_nights))

print("\nMissing values:")
print(combined_df.isna().sum())

print("\nSleep-stage distribution:")
print(combined_df["sleep_stage"].value_counts().sort_index())

display(combined_df.head())

Combined shape: (205013, 13)
Participants: 47
Total nights: 253
Failed nights: 0

Missing values:
epoch          0
sleep_stage    0
hr_mean        0
hr_std         0
hr_min         0
hr_max         0
hr_count       0
motion_mean    0
motion_std     0
motion_min     0
motion_max     0
person         0
night          0
dtype: int64

Sleep-stage distribution:
sleep_stage
0    22266
1    14120
2    76500
3    38742
4    53385
Name: count, dtype: int64


,epoch,sleep_stage,hr_mean,hr_std,hr_min,hr_max,hr_count,motion_mean,motion_std,motion_min,motion_max,person,night
0,0,0,64.333333,1.966384,63.0,68.0,6,1.005263,0.083865,0.597759,1.707987,Bidslab00,1
1,1,0,69.333333,1.527525,68.0,71.0,3,1.016832,0.136838,0.334249,2.314065,Bidslab00,1
2,2,0,74.000000,1.000000,73.0,75.0,3,1.048849,0.246704,0.263571,3.822546,Bidslab00,1
3,3,0,76.666667,3.214550,73.0,79.0,3,1.052265,0.263902,0.115942,3.001449,Bidslab00,1
4,4,0,72.000000,0.000000,72.0,72.0,3,1.032463,0.199030,0.374140,2.522472,Bidslab00,1


In [22]:
combined_df.to_csv("bidsleep_processed.csv", index=False)

In [23]:
stage_map = {
    0: 0,  # Wake
    1: 1,  # N1 -> Light
    2: 1,  # N2 -> Light
    3: 2,  # Deep
    4: 3   # REM
}

combined_df["stage_4"] = combined_df["sleep_stage"].map(stage_map)

print(combined_df["stage_4"].value_counts().sort_index())

stage_4
0    22266
1    90620
2    38742
3    53385
Name: count, dtype: int64


In [24]:
combined_df.to_csv("bidsleep_processed.csv", index=False)

In [25]:
from sklearn.model_selection import train_test_split

# Get unique participants
persons = combined_df["person"].unique()

# 70% train, 30% remaining
train_persons, temp_persons = train_test_split(
    persons,
    test_size=0.30,
    random_state=42
)

# Split remaining participants into validation and test
val_persons, test_persons = train_test_split(
    temp_persons,
    test_size=0.50,
    random_state=42
)

print("Training participants:", len(train_persons))
print("Validation participants:", len(val_persons))
print("Testing participants:", len(test_persons))

Training participants: 32
Validation participants: 7
Testing participants: 8


In [26]:
train_df = combined_df[
    combined_df["person"].isin(train_persons)
].copy()

val_df = combined_df[
    combined_df["person"].isin(val_persons)
].copy()

test_df = combined_df[
    combined_df["person"].isin(test_persons)
].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (139780, 14)
Validation: (25676, 14)
Test: (39557, 14)


In [27]:
print(
    "Train & Validation:",
    set(train_persons) & set(val_persons)
)

print(
    "Train & Test:",
    set(train_persons) & set(test_persons)
)

print(
    "Validation & Test:",
    set(val_persons) & set(test_persons)
)

Train & Validation: set()
Train & Test: set()
Validation & Test: set()


In [28]:
stage_names = {
    0: "Wake",
    1: "Light",
    2: "Deep",
    3: "REM"
}

for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}:")

    distribution = (
        data["stage_4"]
        .map(stage_names)
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print(distribution)


Train:
stage_4
Light    44.18
REM      26.19
Deep     18.95
Wake     10.69
Name: proportion, dtype: float64

Validation:
stage_4
Light    45.69
REM      25.31
Deep     18.25
Wake     10.75
Name: proportion, dtype: float64

Test:
stage_4
Light    43.33
REM      25.99
Deep     19.14
Wake     11.55
Name: proportion, dtype: float64


# Saving the Splitted and Processed DataFrames

In [29]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)